In [ ]:
# 0. Environment Setup

# Clone the gemma repository
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository for UI/UX
!git clone https://github.com/google-deepmind/dialog.git || true
!pip install -q ./dialog

# Ensure local modules are in path
import sys
import os
sys.path.append(os.getcwd())

In [ ]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

In [ ]:
import importlib
import jax
import os
import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from gemma import gm

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"

# jax.config.update("jax_disable_jit", True) # Temporarily disable JIT to bypass the hashing error

In [ ]:
import dataclasses
inference_config = Gemma_Titans_Config(
    **{f.name: getattr(Gemma3_1B_Titans.config, f.name) for f in dataclasses.fields(Gemma_Titans_Config) if f.name != 'is_training_mode'},
    is_training_mode=False # <--- ВЫКЛЮЧАЕМ ТРЕНИРОВКУ ДЛЯ СЭМПЛЕРА
)

In [ ]:
import google.colab
import shutil
import orbax.checkpoint as ocp

if os.path.exists('/content/Titans_jax/saved_titans_delta.zip'):
    shutil.unpack_archive('/content/Titans_jax/saved_titans_delta.zip',"./saved_titans_delta")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(load_path)

import jax.numpy as jnp
# gemma_config = Gemma3_1B_Titans.config
# inference_config = Gemma_Titans_Config(
#     **{f.name: getattr(Gemma3_1B_Titans.config, f.name) for f in dataclasses.fields(_config.TransformerConfig)},
#     is_training_mode=False # <--- ВЫКЛЮЧАЕМ ТРЕНИРОВКУ ДЛЯ СЭМПЛЕРА
# )

# gemma_config.is_training_mode = False

model = Gemma3_1B_Titans(
    config=inference_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

print("Loading weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
loaded_titans_params = load_titans_weights("./saved_titans_delta")

# Снимаем обертку Kauldron, если она есть
if 'model' in loaded_titans_params: # and 'blocks' not in loaded_titans_params:
    print("Removing Kauldron 'model' wrapper from loaded weights...")
    loaded_titans_params = loaded_titans_params['model']

merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

# Жесткая проверка: убеждаемся, что веса реально вшились в рабочую ветку
assert 'memory_gate' in merged_params['layer_5'], "FATAL: Titans weights failed to merge!"
print("Weights merged successfully!")


tokenizer = gm.text.Gemma3Tokenizer()

In [ ]:
merged_params['layer_5'].keys()

## 4. Interactive Dialogue with `google-deepmind/dialog`

In [ ]:
import dialog
from gemma import gm
import jax

# Initialize Sampler and Conversation
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

conv = dialog.Conversation()


def chat(user_input: str):
    global conv
    # Add user message
    conv += dialog.User(user_input)

    # Convert conversation to prompt (Gemma 3 format)
    prompt = conv.as_text(training=False)

    # Generate response
    # Note: Sampler handles the caching of Titans memory automatically
    response_text = sampler.sample(prompt, max_new_tokens=128)

    # Add model response to UI
    conv += dialog.Model(response_text)
    conv.show()

# Example usage:
# chat("Привет! Кто такие Титаны в мифологии?")

In [ ]:
chat("Who is Polyphemus in the ancient greek mythology?")
